# Diabetes Classification - Multi-Model Training & EDA
Run this notebook in Google Colab to train the Logistic Regression and XGBoost models, and generate all Exploratory Data Analysis (EDA) graphs. Make sure to upload the dataset CSV (`diabetes_binary_5050split_health_indicators_BRFSS2015.csv`) to the Colab files section first!

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn xgboost joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, auc, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import joblib
import time

# Set seaborn style for beautiful plots
sns.set_theme(style="whitegrid")

print('Loading dataset...')
# Assuming the CSV is uploaded directly to the Colab root directory
df = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')

X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

## 1. Model Training & Tuning

In [ ]:
print('\n--- Tuning Logistic Regression ---')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'scaler.pkl')

logreg = LogisticRegression(max_iter=1000)
param_grid_lr = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2']}
random_search_lr = RandomizedSearchCV(estimator=logreg, param_distributions=param_grid_lr, n_iter=4, scoring='roc_auc', cv=cv, verbose=1, random_state=42, n_jobs=-1)
random_search_lr.fit(X_train_scaled, y_train)

best_lr = random_search_lr.best_estimator_
joblib.dump(best_lr, 'best_logreg_model.pkl')
lr_y_pred_proba = best_lr.predict_proba(X_test_scaled)[:, 1]
lr_test_auc = roc_auc_score(y_test, lr_y_pred_proba)
print(f'Logistic Regression Final Test AUC: {lr_test_auc:.4f}')

print('\n--- Tuning XGBoost ---')
xgb = XGBClassifier(tree_method='hist', eval_metric='logloss')
param_grid_xgb = {'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'n_estimators': [100, 200]}
random_search_xgb = RandomizedSearchCV(estimator=xgb, param_distributions=param_grid_xgb, n_iter=4, scoring='roc_auc', cv=cv, verbose=1, random_state=42, n_jobs=-1)
random_search_xgb.fit(X_train, y_train)

best_xgb = random_search_xgb.best_estimator_
joblib.dump(best_xgb, 'best_xgboost_model.pkl')

xgb_y_pred_proba = best_xgb.predict_proba(X_test)[:, 1]
xgb_test_auc = roc_auc_score(y_test, xgb_y_pred_proba)
print(f'XGBoost Final Test AUC: {xgb_test_auc:.4f}')

print('\nTraining Complete! Download the generated .pkl files from the Colab file explorer to use in your web app.')

## 2. Exploratory Data Analysis & Visualizations

In [ ]:
# 1. Class Imbalance Check (Target Distribution)
plt.figure(figsize=(6, 5))
ax = sns.countplot(x='Diabetes_binary', data=df, palette='Set2')
plt.title('Target Variable Distribution (Class Balance Check)', fontweight='bold')
plt.xlabel('Diabetes Status (0 = No, 1 = Yes)')
plt.ylabel('Patient Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='baseline', fontsize=11, color='black', xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Correlation Heatmap
plt.figure(figsize=(16, 12))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3. BMI vs Diabetes Distribution
plt.figure(figsize=(8, 6))
sns.violinplot(x='Diabetes_binary', y='BMI', data=df, palette='muted')
plt.title('Distribution of BMI by Diabetes Status', fontweight='bold')
plt.xlabel('Diabetes Status (0 = No, 1 = Yes)')
plt.ylabel('Body Mass Index (BMI)')
plt.tight_layout()
plt.show()

In [ ]:
# 4. ROC Curves Comparison
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_y_pred_proba)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC = {lr_test_auc:.4f})', color='blue')
plt.plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUC = {xgb_test_auc:.4f})', color='darkorange')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 5. Confusion Matrices
y_pred_lr = best_lr.predict(X_test_scaled)
y_pred_xgb = best_xgb.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['No Diabetes', 'Diabetes'])
disp_lr.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Logistic Regression Confusion Matrix', fontweight='bold')
axes[0].grid(False)

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['No Diabetes', 'Diabetes'])
disp_xgb.plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title('XGBoost Confusion Matrix', fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.show()